# Gemma 2 (2B) — Qatar Department Router

Fine-tune **Gemma 2 2B** on `data/routing_dataset.jsonl` to output JSON routing tags only:

- `visa`, `residency`, `traffic`, `general`

**Example output:** `{"department": "visa"}`

Upload before training:
- `routing_config.py`
- `data/routing_dataset.jsonl`
- `test_questions.json`

### Installation

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    !uv pip install -qqq torch torchvision bitsandbytes "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers trl unsloth unsloth_zoo
!uv pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0"

### Load Gemma 2 model

In [ ]:
from unsloth import FastLanguageModel
import torch

from routing_config import MODEL_NAME

max_seq_length = 1024
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

### Add LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

<a name="Data"></a>
### Data prep
Load `data/routing_dataset.jsonl` and format with the Gemma Alpaca template.

In [ ]:
import json
from pathlib import Path

from datasets import Dataset

from routing_config import DATASET_PATH, format_router_prompt

dataset_path = Path(DATASET_PATH)
if not dataset_path.exists():
    raise FileNotFoundError(f"Missing dataset: {dataset_path}")

rows = []
with dataset_path.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

print(f"Loaded {len(rows)} examples")

def formatting_prompts_func(examples):
    texts = []
    for query, department in zip(examples["query"], examples["department"]):
        texts.append(format_router_prompt(tokenizer, query, department.strip().lower(), include_eos = True))
    return {"text": texts}

dataset = Dataset.from_list(rows)
dataset = dataset.map(formatting_prompts_func, batched = True)
print(dataset[0]["text"][:600])

### Pre-training inference check

In [ ]:
from routing_config import configure_tokenizer_for_generation, generate_router_completion, parse_department_tag

FastLanguageModel.for_inference(model)
configure_tokenizer_for_generation(tokenizer)

sample_query = "I want to apply for visa"
generated = generate_router_completion(model, tokenizer, sample_query)
print("Raw:", repr(generated))
print("Parsed tag:", parse_department_tag(generated))
print(
    "\nNote: Before training, the base model often emits non-JSON text or garbage; "
    "parsed tag should become reliable after SFT."
)

<a name="Train"></a>
### Train

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs_router",
        report_to = "none",
    ),
)

trainer_stats = trainer.train()
trainer_stats.metrics

<a name="Inference"></a>
### Inference

In [ ]:
def route_query(query: str) -> dict:
    FastLanguageModel.for_inference(model)
    raw = generate_router_completion(model, tokenizer, query).strip()
    return {"department": parse_department_tag(raw), "raw": raw}

for q in [
    "I want to apply for visa",
    "What are your office hours?",
    "The camera on Salwa Road caught me yesterday.",
]:
    print(q, "->", route_query(q))

### Evaluate on 15 tricky routing questions

In [ ]:
with open("test_questions.json", "r", encoding="utf-8") as f:
    test_questions = json.load(f)

correct = 0
print(f"{'ID':<4} {'Expected':<10} {'Predicted':<10} OK   Query")
print("-" * 90)
for item in test_questions:
    result = route_query(item["query"])
    predicted = result["department"]
    ok = predicted == item["expected"]
    correct += int(ok)
    print(f"{item['id']:<4} {item['expected']:<10} {str(predicted):<10} {str(ok):<4} {item['query'][:55]}")

print(f"\nAccuracy: {correct}/{len(test_questions)} ({correct/len(test_questions):.1%})")

### Save LoRA adapters

In [ ]:
from routing_config import LORA_DIR

model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f"Saved LoRA adapters to {LORA_DIR}")

### Compare base vs fine-tuned locally

```bash
python compare_models.py --lora-path gemma_router_lora
```